In [0]:
%sql 

CREATE VOLUME IF NOT EXISTS mlpractice.source.dataanalysis;



In [0]:
import urllib.request as urllib


files = ["2019.csv", "2020.csv", "2021.csv"]
base_volume = "/Volumes/mlpractice/source/dataanalysis"

def upload_to_volume(file_names: list[str], distination_volume: str):
    
    url = "https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/refs/heads/main/DataAnalytics"
    for file in files:
        urllib.urlretrieve(f"{url}/{file}", f"{distination_volume}/{file}")
    
    print(f"Files uploaded to {distination_volume}")



In [0]:
upload_to_volume(files, base_volume)

In [0]:
# Read the files from the volume as a cloudFiles
df = (
    spark.read.load(f"{base_volume}/*.csv", format="csv", header=True)
)

display(df.limit(10))

In [0]:
from pyspark.sql import types as T
from pyspark.sql import functions as F

# Define the schema that is suitable for the data

orderSchema = T.StructType([
    T.StructField("SalesOrderNumber", T.StringType()),
    T.StructField("SalesOrderLineNumber", T.IntegerType()),
    T.StructField("OrderDate", T.DateType()),
    T.StructField("CustomerName", T.StringType()),
    T.StructField("Email", T.StringType()),
    T.StructField("Item", T.StringType()),
    T.StructField("Quantity", T.IntegerType()),
    T.StructField("UnitPrice", T.FloatType()),
    T.StructField("Tax", T.FloatType())
])

In [0]:
# Read the files from the volume as a cloudFiles
df_sales = spark.read.load(
    f"{base_volume}/*.csv",
    format="csv",
    schema=orderSchema
)


display(df_sales.limit(10)) 

In [0]:
# SQL query from temporary view 
df_sales.createOrReplaceTempView("salesorders")

In [0]:
sql_query = """
    SELECT
        CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear,
        SUM(UnitPrice * Quantity) AS GrossRevenue
    FROM salesorders
    GROUP BY CAST(YEAR(OrderDate) AS CHAR(4) )
    ORDER BY OrderYear
"""

df_spark = spark.sql(sql_query)
df_spark.show()

In [0]:
# Data Visualization
from matplotlib import pyplot as plt 

df_pd = df_spark.toPandas()

plt.bar(x=df_pd["OrderYear"], height=df_pd["GrossRevenue"])
plt.show()

In [0]:
import seaborn as sns
# clear the plot area
plt.clf()
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_pd)

**Data Cleaning**


In [0]:
df = df_sales.dropDuplicates()
df = df.withColumn("Tax", F.col("UnitPrice") * 0.08).withColumn("Tax", F.col('Tax').cast('float'))

display(df.limit(10))#


In [0]:
# Create a new DataFrame 

cust_df = df.select("CustomerName", "Email", "Item", "Quantity")
cust_df = cust_df.withColumn("FirstName", F.split(F.col("CustomerName"), " ")[0])
cust_df = cust_df.withColumn("LastName", F.split(F.col("CustomerName"), " ")[1])

display(cust_df.limit(10))

In [0]:
# counting customers and distinct coustomers 
print(cust_df.count())
print(cust_df.distinct().count())

In [0]:
# Create a products sales dataframe 
prod_sales = df.select("Item", "Quantity").groupBy("Item").sum()
display(prod_sales.limit(10))

In [0]:
# Aggregate yearly sales 
yearly_sales = df.select(F.year("OrderDate").alias("SalesYear")).groupBy("SalesYear").count().orderBy("SalesYear")
display(yearly_sales)

**Create a delta table from the dataframe**

In [0]:
df.write.format("delta").mode("append").saveAsTable("mlpractice.bronze.sales")

In [0]:
%skip
# Upload products file 

dbutils.fs.mkdirs("/Volumes/mlpractice/source/dataanalysis/products")

In [0]:
# Upload Products 
files = ["products.csv"]

upload_to_volume(files, f"{base_volume}/products")

In [0]:
df_prod = spark.read.load(f"{base_volume}/products/*.csv", format="csv", header="true", inferSchema="true")
display(df_prod.limit(10))

In [0]:
# save it in bronze layr 
df_prod.write.format("delta").mode("append").saveAsTable("mlpractice.bronze.products")


In [0]:
# Manipulating the delta table by creating a deltatable object

from delta.tables import *

# Create a delta table object 
deltaTable = DeltaTable.forName(spark, "mlpractice.bronze.products")

# Update the table 
deltaTable.update(
    condition = "ProductId = '771'",
    set = {"ListPrice": "ListPrice * 0.9"}
)
# View the updated data as a dataframe 
deltaTable.toDF().show(10)

In [0]:
# Explor logging for the delta table 
deltaTable.history(10).show(10, False, True)

In [0]:
# Creating a dataFrame from the delta dataset
new_df_prod = spark.table('mlpractice.bronze.products')

new_df_prod.show()


*Medallion Architecture with delta lake*

In [0]:
# Load data from git repo to volume. 

#dbutils.fs.mkdirs("/Volumes/mlpractice/source/dataanalysis/sales_edited")
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]
upload_to_volume(files, f"{base_volume}/sales_edited")

In [0]:
# create a dataframe 
df_sales_edited = spark.read.load(
    f"{base_volume}/sales_edited/*.csv", 
    format="csv", 
    schema=orderSchema
)


display(df_sales_edited.limit(10))

In [0]:
%sql
DROP TABLE IF EXISTS mlpractice.bronze.sales;

In [0]:
# Store sales data as a delta table in unity catalog 

df_sales_edited.write.format("delta").mode("append").saveAsTable("mlpractice.bronze.sales_orders_bronze")

# Data Transformation
    Bronze -> Silver 
    

In [0]:
# Create a silver layer dataframe 
silver_df = spark.table("mlpractice.bronze.sales_orders_bronze")

silver_df = silver_df.dropDuplicates()
silver_df = silver_df.withColumn("Tax", F.col('UnitPrice') * 0.08).withColumn("Tax", F.col("Tax").cast('float'))


silver_df.write.format('delta').mode('overwrite').saveAsTable('mlpractice.silver.sales_orders_silver')

# Creating Gold Layer data
    silver -> gold

In [0]:
gold_df = spark.read.table("mlpractice.silver.sales_orders_silver")
display(gold_df.limit(10))

In [0]:
# Aggregaring yearly sales 
yearly_sales = (
    gold_df.select( F.year('OrderDate').alias('SalesYear') )
        .groupBy('SalesYear')
        .count()
        .orderBy('SalesYear')
)
display(yearly_sales)

In [0]:
yearly_sales.write.format('delta').mode('overwrite').saveAsTable('mlpractice.gold.sales_orders_gold')

In [0]:
%sql
select * from mlpractice.gold.sales_orders_gold